In [26]:
import os
import json
import random
from pathlib import Path

In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm

In [28]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import albumentations as A
from albumentations.pytorch import ToTensorV2


In [29]:
from sklearn.model_selection import train_test_split

In [30]:
DATA_ROOT = Path("dataset")
ODIR_ROOT = DATA_ROOT / "ODIR-5K/ODIR-5K"   # путь до внутренней папки

TRAIN_IMG_DIR = ODIR_ROOT / "Training Images"
TEST_IMG_DIR  = ODIR_ROOT / "Testing Images"

# Аннотации (выберите один из файлов, который содержит все метки)
ANNOTATION_FILE = DATA_ROOT / "full_df.csv"   # или data.xlsx

PROCESSED_DIR = Path("processed")
PROCESSED_DIR.mkdir(exist_ok=True)

In [31]:
print("Train images dir exists:", TRAIN_IMG_DIR.exists())
print("Test images dir exists:", TEST_IMG_DIR.exists())
print("Annotation file exists:", ANNOTATION_FILE.exists())

Train images dir exists: True
Test images dir exists: True
Annotation file exists: True


In [32]:
CLASSES = ["N", "D", "G", "C", "A", "H", "M", "O"]
CLASS_NAMES = {
    "N": "Normal",
    "D": "Diabetes",
    "G": "Glaucoma",
    "C": "Cataract",
    "A": "AMD",
    "H": "Hypertension",
    "M": "Myopia",
    "O": "Other",
}

In [33]:
IMG_SIZE    = 224          
BATCH_SIZE  = 32
NUM_WORKERS = 4
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

In [34]:
def load_annotations(ann_file: Path) -> pd.DataFrame:
    """
    Читает Excel-файл ODIR-5K и возвращает DataFrame с колонками:
        filename_left, filename_right, N, D, G, C, A, H, M, O
    """
    print(f"[1/5] Загрузка аннотаций из {ann_file.name} ...")
    df = pd.read_excel(ann_file)

    # Приводим имена колонок к нижнему регистру для удобства
    df.columns = [c.strip() for c in df.columns]

    # Стандартные имена колонок в датасете:
    #   ID, Patient Age, Patient Sex,
    #   Left-Fundus, Right-Fundus,
    #   Left-Diagnostic Keywords, Right-Diagnostic Keywords,
    #   N, D, G, C, A, H, M, O
    rename_map = {
        "Left-Fundus":  "filename_left",
        "Right-Fundus": "filename_right",
    }
    df.rename(columns=rename_map, inplace=True)

    # Убеждаемся что метки — int (0 или 1)
    for cls in CLASSES:
        if cls in df.columns:
            df[cls] = df[cls].astype(int)

    print(f"    Загружено {len(df)} записей (пациентов).")
    return df

In [35]:
def expand_to_images(df: pd.DataFrame, img_dir: Path) -> pd.DataFrame:
    """
    Разворачивает DataFrame из «пар глаз» в список отдельных снимков.
    Каждый снимок получает те же метки, что и пациент.
    Итого строк ≈ 2×len(df).
    """
    print("[2/5] Разворачивание пар снимков в единый список ...")
    rows = []
    for _, row in df.iterrows():
        for side in ("left", "right"):
            fname = row[f"filename_{side}"]
            fpath = img_dir / fname
            rows.append({
                "filepath": str(fpath),
                "filename": fname,
                "side": side,
                **{cls: row[cls] for cls in CLASSES if cls in row},
            })
    result = pd.DataFrame(rows)
    # Фильтруем отсутствующие файлы
    exists_mask = result["filepath"].apply(lambda p: Path(p).exists())
    missing = (~exists_mask).sum()
    if missing:
        print(f"    ⚠️  Пропущено {missing} отсутствующих файлов.")
    result = result[exists_mask].reset_index(drop=True)
    print(f"    Итого снимков: {len(result)}")
    return result


In [36]:
def analyze_class_distribution(df: pd.DataFrame, save_dir: Path):
    """Строит и сохраняет графики распределения классов."""
    print("[3/5] Анализ распределения классов ...")

    counts = {CLASS_NAMES[c]: int(df[c].sum()) for c in CLASSES if c in df.columns}
    total  = len(df)

    print("\n    Количество снимков по классам:")
    print(f"    {'Класс':<30} {'Кол-во':>7}  {'%':>6}")
    print("    " + "-" * 46)
    for name, cnt in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"    {name:<30} {cnt:>7}  {cnt/total*100:>5.1f}%")

    # Многоклассовость: сколько снимков имеют >1 метки
    if all(c in df.columns for c in CLASSES):
        multi = (df[CLASSES].sum(axis=1) > 1).sum()
        print(f"\n    Мультиметочных снимков: {multi} ({multi/total*100:.1f}%)")
        print(f"    Однометочных снимков:   {total-multi} ({(total-multi)/total*100:.1f}%)")

    # График
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("ODIR-5K — распределение классов", fontsize=14, y=1.02)

    # Bar chart
    ax = axes[0]
    names  = list(counts.keys())
    values = list(counts.values())
    colors = ["#1D9E75", "#BA7517", "#3B6D11", "#185FA5",
              "#993C1D", "#993556", "#534AB7", "#5F5E5A"]
    bars = ax.barh(names, values, color=colors, alpha=0.85)
    ax.set_xlabel("Количество снимков")
    ax.set_title("Абсолютное количество")
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                str(val), va="center", fontsize=9)
    ax.set_xlim(0, max(values) * 1.15)

    # Pie chart
    ax2 = axes[1]
    wedges, texts, autotexts = ax2.pie(
        values, labels=names, colors=colors, autopct="%1.1f%%",
        startangle=90, pctdistance=0.8
    )
    ax2.set_title("Доля каждого класса")

    plt.tight_layout()
    out = save_dir / "class_distribution.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"\n    График сохранён → {out}")

    return counts

In [37]:
def analyze_image_sizes(df: pd.DataFrame, save_dir: Path, sample_n: int = 200):
    """Проверяет реальные размеры снимков на выборке."""
    print("\n    Проверка размеров снимков (выборка) ...")
    sample = df["filepath"].sample(min(sample_n, len(df)), random_state=RANDOM_SEED)
    sizes  = []
    for path in tqdm(sample, desc="    чтение", leave=False):
        try:
            with Image.open(path) as img:
                sizes.append(img.size)  # (width, height)
        except Exception:
            pass

    if not sizes:
        print("    Не удалось открыть снимки. Проверьте пути.")
        return

    widths  = [s[0] for s in sizes]
    heights = [s[1] for s in sizes]
    print(f"    Ширина:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}")
    print(f"    Высота:  min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}")

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.scatter(widths, heights, alpha=0.4, s=20, color="#185FA5")
    ax.set_xlabel("Ширина (px)")
    ax.set_ylabel("Высота (px)")
    ax.set_title("Размеры снимков в датасете")
    out = save_dir / "image_sizes.png"
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.close()
    print(f"    График сохранён → {out}")

In [38]:
def split_dataset(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Стратифицированное разделение: 70% train / 15% val / 15% test.
    Стратификация по основному классу (argmax меток).
    """
    print("[4/5] Разбивка на train / val / test ...")

    label_cols = [c for c in CLASSES if c in df.columns]

    # Псевдоскаляр для стратификации: argmax меток (0..7)
    # Если все метки 0 — помечаем как "Unknown"
    def primary_label(row):
        vals = row[label_cols].values
        if vals.sum() == 0:
            return -1
        return int(np.argmax(vals))

    df = df.copy()
    df["_strat"] = df.apply(primary_label, axis=1)

    train_val, test = train_test_split(
        df, test_size=0.15, random_state=RANDOM_SEED,
        stratify=df["_strat"]
    )
    train, val = train_test_split(
        train_val, test_size=0.15 / 0.85, random_state=RANDOM_SEED,
        stratify=train_val["_strat"]
    )

    train = train.drop(columns=["_strat"]).reset_index(drop=True)
    val   = val.drop(columns=["_strat"]).reset_index(drop=True)
    test  = test.drop(columns=["_strat"]).reset_index(drop=True)

    print(f"    Train: {len(train):>5} снимков")
    print(f"    Val:   {len(val):>5} снимков")
    print(f"    Test:  {len(test):>5} снимков")
    return train, val, test

In [39]:
def get_train_transforms(img_size: int = IMG_SIZE) -> A.Compose:
    """
    Аугментации для тренировочной выборки.
    Медицинские снимки: осторожно с цветом, допустимы повороты и отражения.
    """
    return A.Compose([
        # Базовый ресайз
        A.Resize(img_size, img_size),

        # Геометрические трансформации
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.5),

        # Имитация разных условий съёмки
        A.RandomBrightnessContrast(
            brightness_limit=0.2, contrast_limit=0.2, p=0.5
        ),
        A.HueSaturationValue(
            hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3
        ),

        # Имитация шума камеры
        A.GaussNoise(var_limit=(10, 50), p=0.2),

        # Случайное обрезание (симулирует разное поле зрения)
        A.RandomResizedCrop(
            size=(img_size, img_size), scale=(0.85, 1.0), p=0.3
        ),

        # Нормализация ImageNet (т.к. используем pretrained)
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2(),
    ])

In [40]:
def get_val_transforms(img_size: int = IMG_SIZE) -> A.Compose:
    """Трансформации для val/test — только ресайз и нормализация."""
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2(),
    ])

In [41]:
class ODIRDataset(Dataset):
    """
    PyTorch Dataset для ODIR-5K.

    Параметры:
        df         — DataFrame с колонками filepath + метки
        transforms — albumentations-трансформации
        classes    — список меток (по умолчанию CLASSES)
    """

    def __init__(
        self,
        df: pd.DataFrame,
        transforms: A.Compose,
        classes: list[str] = CLASSES,
    ):
        self.df         = df.reset_index(drop=True)
        self.transforms = transforms
        self.classes    = [c for c in classes if c in df.columns]

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        row = self.df.iloc[idx]

        # Загрузка снимка
        try:
            image = np.array(Image.open(row["filepath"]).convert("RGB"))
        except Exception as e:
            # Возвращаем чёрный снимок при ошибке чтения
            print(f"Ошибка чтения {row['filepath']}: {e}")
            image = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

        # Применяем трансформации
        augmented = self.transforms(image=image)
        image_t   = augmented["image"]  # Tensor [3, H, W]

        # Метки: float32 для BCEWithLogitsLoss
        labels = torch.tensor(
            row[self.classes].values.astype(np.float32),
            dtype=torch.float32
        )
        return image_t, labels


In [42]:
def compute_class_weights(df: pd.DataFrame) -> torch.Tensor:
    """
    Вычисляет pos_weight для BCEWithLogitsLoss:
        pos_weight[i] = (кол-во негативных) / (кол-во позитивных)
    Чем реже класс — тем выше его вес.
    """
    label_cols = [c for c in CLASSES if c in df.columns]
    pos_counts = df[label_cols].sum(axis=0).values  # кол-во 1
    neg_counts = len(df) - pos_counts                # кол-во 0
    weights    = neg_counts / (pos_counts + 1e-8)    # +eps чтобы не делить на 0
    print("\n    Веса классов (pos_weight для BCEWithLogitsLoss):")
    for cls, w in zip(label_cols, weights):
        print(f"      {CLASS_NAMES[cls]:<30}: {w:.2f}")
    return torch.tensor(weights, dtype=torch.float32)


In [45]:
def main():
    print("=" * 60)
    print("  ODIR-5K Data Preparation")
    print("=" * 60)

    # 1. Загрузка аннотаций
    if not TRAIN_ANN_FILE.exists():
        print(f"\n⛔  Файл аннотаций не найден: {ANNOTATION_FILE}")
        print("    Скачайте датасет с Kaggle:")
        print("    https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k")
        print("    и распакуйте в папку ./data/")
        return

    df_raw = load_annotations(ANNOTATION_FILE)

    # 2. Разворачиваем пары снимков
    df = expand_to_images(df_raw, TRAIN_IMG_DIR)

    # 3. Анализ
    analyze_class_distribution(df_raw, PROCESSED_DIR)   # по пациентам (не по снимкам)
    analyze_image_sizes(df, PROCESSED_DIR)

    # 4. Разбивка
    train_df, val_df, test_df = split_dataset(df)

    # Сохраняем CSV для последующих шагов
    train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
    val_df.to_csv(PROCESSED_DIR / "val.csv", index=False)
    test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)
    print(f"\n    CSV сохранены в {PROCESSED_DIR}/")

    # 5. Создаём Dataset и DataLoader
    print("\n[5/5] Создание DataLoader ...")
    train_transforms = get_train_transforms()
    val_transforms   = get_val_transforms()

    train_dataset = ODIRDataset(train_df, train_transforms)
    val_dataset   = ODIRDataset(val_df,   val_transforms)
    test_dataset  = ODIRDataset(test_df,  val_transforms)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    # Быстрая проверка — один батч
    print("\n    Проверка одного батча ...")
    images, labels = next(iter(train_loader))
    print(f"    images.shape : {images.shape}   (B, C, H, W)")
    print(f"    labels.shape : {labels.shape}  (B, num_classes)")
    print(f"    labels dtype : {labels.dtype}")
    print(f"    Пример меток:\n{labels[:3]}")

    # 6. Веса классов
    pos_weight = compute_class_weights(train_df)

    # Сохраняем конфиг для следующего скрипта (обучение)
    config = {
        "img_size":    IMG_SIZE,
        "batch_size":  BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "classes":     CLASSES,
        "class_names": CLASS_NAMES,
        "processed_dir": str(PROCESSED_DIR),
        "pos_weight":  pos_weight.tolist(),
    }
    with open(PROCESSED_DIR / "config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)
    print(f"\n    Конфиг сохранён → {PROCESSED_DIR}/config.json")

    print("\n" + "=" * 60)
    print("  ✅  Подготовка данных завершена!")
    print(f"     Следующий шаг → 02_train_model.py")
    print("=" * 60)

    return train_loader, val_loader, test_loader, pos_weight

In [46]:
if __name__ == "__main__":
    main()


  ODIR-5K Data Preparation

⛔  Файл аннотаций не найден: dataset\full_df.csv
    Скачайте датасет с Kaggle:
    https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k
    и распакуйте в папку ./data/
